# Bayesian neural networks for flood-hazard uncertainty: VI vs MCMC

This notebook compares two ways of approximating the posterior of the same Bayesian MLP (inputs → 64 → 32 → 4 classes). The data is a gridded surface-water flood-hazard dataset for Exeter, UK, with three return periods: 1-in-30, 1-in-100 and 1-in-1000 years.

- **VI-BNN** uses mean-field Gaussian weights trained on the ELBO (reparameterisation trick, KL annealing). Predictions average 50 Monte Carlo forward passes.
- **MCMC-BNN** puts N(0, 1) priors on all weights and samples the posterior with Pyro's No-U-Turn Sampler (NUTS). Predictions average 150 posterior samples.

Class 0 (no hazard) covers 93–99% of cells, depending on the return period, so accuracy alone says little. The evaluation therefore focuses on proper scoring rules (NLL, Brier), calibration (ECE) and uncertainty quality.

**Runtime warning:** a full run takes several hours on CPU. NUTS took 80–110 minutes per hazard and VI about 1–2 minutes per hazard. For a quick end-to-end check, set `SMOKE_TEST = True` in the configuration cell.

**Setup:** run `pip install -r requirements.txt` and place the dataset at `data/Flood_Data.csv` (see the README). Outputs are written to `outputs/`. The results of the full run are kept in `results/`.

## 1. Imports

In [ ]:
import os
import time
import warnings
from typing import Any, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

import pyro
import pyro.distributions as dist
from pyro.infer import MCMC, NUTS

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    log_loss, brier_score_loss
)
from imblearn.combine import SMOTETomek

warnings.filterwarnings('ignore')


## 2. Configuration

Hyperparameters match the reported run. `SMOKE_TEST` shrinks the epochs and MCMC samples for a fast check.

In [ ]:
# Global configuration
SEED = 50
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Paths are relative to the repository root (see README for the data file)
CSV_PATH = "data/Flood_Data.csv"
OUTPUT_DIR = "outputs"

# Experiment parameters
HAZARDS = ["HAZARD 1 IN 30", "HAZARD 1 IN 100", "HAZARD 1 IN 1000"]
NUM_CLASSES = 4  # Classes: 0, 1, 2, 3 (after merging 4→3)

# VI-BNN hyperparameters
VI_EPOCHS = 50
VI_BATCH_SIZE = 256
VI_LEARNING_RATE = 1e-3
VI_PATIENCE = 10
VI_MC_SAMPLES = 50
VI_KL_ANNEAL_EPOCHS = 10  # linear KL warm-up for stability

# MCMC hyperparameters
MCMC_SUBSET_SIZE = 5000  # NUTS runs on a subsample of the original (pre-SMOTE) training data
MCMC_WARMUP = 200
MCMC_SAMPLES = 150

# Architecture
HIDDEN_DIM_1 = 64
HIDDEN_DIM_2 = 32

# Quick end-to-end check: set to True to run with a handful of epochs and MCMC
# samples (about 7 minutes on CPU). The resulting metrics are not meaningful.
SMOKE_TEST = False
if SMOKE_TEST:
    VI_EPOCHS = 2
    VI_MC_SAMPLES = 5
    MCMC_SUBSET_SIZE = 500
    MCMC_WARMUP = 5
    MCMC_SAMPLES = 5

# Set seeds
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
pyro.set_rng_seed(SEED)

print(f"\n{'='*80}")
print("BAYESIAN NEURAL NETWORK FLOOD HAZARD EXPERIMENT")
print(f"{'='*80}")
print(f"\n✓ Device: {DEVICE}")
print(f"✓ Seed: {SEED}")
print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ Pyro version: {pyro.__version__}")


## 3. Load and preprocess the data

Rows with missing values are dropped (67,330 → 65,528 grid cells). Hazard class 4 is merged into class 3, because it has a single cell for the 1-in-30 target. This gives four ordinal classes for each return period.

In [ ]:
def load_and_preprocess_data(csv_path: str) -> pd.DataFrame:
    """
    Load flood risk CSV and merge class 4 into class 3 for all hazard columns.

    Parameters:
    -----------
    csv_path : str
        Path to flood risk CSV file

    Returns:
    --------
    df : pd.DataFrame
        Preprocessed dataframe with 4-class targets (0, 1, 2, 3)
    """
    print(f"\n{'='*80}")
    print("LOADING AND PREPROCESSING DATA")
    print(f"{'='*80}")

    # Load CSV
    df = pd.read_csv(csv_path)
    print(f"\n✓ Loaded {len(df):,} samples")
    print(f"✓ Features: {df.shape[1]} columns")
    df=df.dropna()
    # Merge class 4 → 3 for all hazard columns
    for hazard_col in HAZARDS:
        if hazard_col in df.columns:
            original_dist = df[hazard_col].value_counts().sort_index()
            print(f"\n✓ {hazard_col} - Original distribution:")
            for cls, count in original_dist.items():
                print(f"    Class {cls}: {count:6d} ({100*count/len(df):5.2f}%)")

            # Merge class 4 into class 3
            df[hazard_col] = df[hazard_col].replace(4, 3)

            # Verify new distribution
            new_dist = df[hazard_col].value_counts().sort_index()
            print(f"\n✓ {hazard_col} - After merging class 4→3:")
            for cls, count in new_dist.items():
                print(f"    Class {cls}: {count:6d} ({100*count/len(df):5.2f}%)")

            # Sanity check: no class 4 should remain
            assert 4 not in df[hazard_col].values, \
                f"Class 4 still present in {hazard_col}!"
            print(f"\n✓ Verified: No class 4 remaining in {hazard_col}")

    return df


## 4. Stratified split, SMOTE-Tomek and scaling

The data is split 60/20/20, stratified by class. SMOTE-Tomek is applied to the training partition only, so validation and test keep the natural class distribution. Features are standardised with a scaler fitted on the resampled training set. The original (pre-SMOTE) training data is kept for MCMC.

In [ ]:
def split_and_resample(
    X: np.ndarray,
    y: np.ndarray,
    test_size: float = 0.2,
    val_size: float = 0.2,
    random_state: int = SEED
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray,
           np.ndarray, np.ndarray]:
    """
    Split data stratified, apply SMOTE-Tomek to training set ONLY.

    Returns both the resampled and the original training data;
    the MCMC model is fitted on the original (pre-SMOTE) data.

    Parameters:
    -----------
    X : np.ndarray
        Feature matrix
    y : np.ndarray
        Target labels
    test_size : float
        Fraction for test set
    val_size : float
        Fraction of (1-test_size) for validation
    random_state : int
        Random seed

    Returns:
    --------
    X_train_res : np.ndarray
        Resampled training features (post-SMOTE-Tomek)
    y_train_res : np.ndarray
        Resampled training labels (post-SMOTE-Tomek)
    X_val : np.ndarray
        Validation features (original, no SMOTE)
    y_val : np.ndarray
        Validation labels
    X_test : np.ndarray
        Test features (original, no SMOTE)
    y_test : np.ndarray
        Test labels
    X_train_orig : np.ndarray
        Original training features (pre-SMOTE)
    y_train_orig : np.ndarray
        Original training labels (pre-SMOTE)
    """
    print(f"\n{'='*80}")
    print("SPLITTING DATA AND APPLYING SMOTE-TOMEK")
    print(f"{'='*80}")

    # Initial train+val / test split
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

    # Train / val split
    val_fraction = val_size / (1 - test_size)
    X_train_orig, X_val, y_train_orig, y_val = train_test_split(
        X_temp, y_temp, test_size=val_fraction, random_state=random_state, stratify=y_temp
    )

    print(f"\n✓ Initial split:")
    print(f"  Train:       {len(X_train_orig):6d} samples ({100*len(X_train_orig)/len(X):.1f}%)")
    print(f"  Validation:  {len(X_val):6d} samples ({100*len(X_val)/len(X):.1f}%)")
    print(f"  Test:        {len(X_test):6d} samples ({100*len(X_test)/len(X):.1f}%)")

    # Display class distribution BEFORE SMOTE
    print(f"\n✓ Training set class distribution (BEFORE SMOTE-Tomek):")
    unique, counts = np.unique(y_train_orig, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:6d} ({100*count/len(y_train_orig):5.2f}%)")

    # Apply SMOTE-Tomek to training set only
    print(f"\n✓ Applying SMOTE-Tomek to training set...")
    smote_tomek = SMOTETomek(random_state=random_state)

    start_time = time.perf_counter()
    X_train_res, y_train_res = smote_tomek.fit_resample(X_train_orig, y_train_orig)
    resample_time = time.perf_counter() - start_time

    print(f"  ✓ SMOTE-Tomek completed in {resample_time:.2f} seconds")
    print(f"  ✓ Training set size: {len(X_train_orig)} → {len(X_train_res)}")

    # Display class distribution AFTER SMOTE
    print(f"\n✓ Training set class distribution (AFTER SMOTE-Tomek):")
    unique, counts = np.unique(y_train_res, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:6d} ({100*count/len(y_train_res):5.2f}%)")

    # Check balance
    balance_ratio = max(counts) / min(counts)
    print(f"\n✓ Balance ratio (max/min): {balance_ratio:.2f}")

    # Standardize features
    scaler = StandardScaler()
    X_train_res = scaler.fit_transform(X_train_res)
    X_train_orig = scaler.transform(X_train_orig)  # Transform original too
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)

    print(f"\n✓ Features standardized (mean=0, std=1)")

    return X_train_res, y_train_res, X_val, y_val, X_test, y_test, X_train_orig, y_train_orig


## 5. PyTorch data loaders

In [ ]:
def create_dataloaders(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    batch_size: int = 256
) -> Tuple[DataLoader, DataLoader, DataLoader]:
    """
    Create PyTorch DataLoaders from numpy arrays.

    Parameters:
    -----------
    X_train, y_train : Training data
    X_val, y_val : Validation data
    X_test, y_test : Test data
    batch_size : Batch size for DataLoader

    Returns:
    --------
    train_loader, val_loader, test_loader : DataLoader objects
    """
    print(f"\n{'='*80}")
    print("CREATING PYTORCH DATALOADERS")
    print(f"{'='*80}")

    # Convert to tensors
    X_train_t = torch.FloatTensor(X_train)
    y_train_t = torch.LongTensor(y_train)
    X_val_t = torch.FloatTensor(X_val)
    y_val_t = torch.LongTensor(y_val)
    X_test_t = torch.FloatTensor(X_test)
    y_test_t = torch.LongTensor(y_test)

    # Create datasets
    train_dataset = TensorDataset(X_train_t, y_train_t)
    val_dataset = TensorDataset(X_val_t, y_val_t)
    test_dataset = TensorDataset(X_test_t, y_test_t)

    # Create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    print(f"\n✓ DataLoaders created:")
    print(f"  Batch size: {batch_size}")
    print(f"  Train batches: {len(train_loader)}")
    print(f"  Val batches:   {len(val_loader)}")
    print(f"  Test batches:  {len(test_loader)}")

    return train_loader, val_loader, test_loader


## 6. Evaluation metrics

The metrics are:

- accuracy;
- negative log-likelihood (log loss);
- multi-class Brier score (mean over one-hot entries);
- top-label expected calibration error (10 bins);
- mean predictive entropy;
- uncertainty–error correlation: the Pearson correlation between predictive entropy and misclassification.

In [ ]:
def evaluate_predictions(
    y_true: np.ndarray,
    probs_samples: np.ndarray,
    probs_mean: np.ndarray,
    num_bins: int = 10
) -> Dict[str, float]:
    """
    Compute comprehensive evaluation metrics for BNN predictions.

    Parameters:
    -----------
    y_true : np.ndarray
        True labels (N,)
    probs_samples : np.ndarray
        Probability samples from BNN (num_samples, N, num_classes)
    probs_mean : np.ndarray
        Mean probabilities (N, num_classes)
    num_bins : int
        Number of bins for ECE calculation

    Returns:
    --------
    metrics : dict
        Dictionary of evaluation metrics
    """
    y_pred = np.argmax(probs_mean, axis=1)

    # Accuracy
    acc = accuracy_score(y_true, y_pred)

    # Negative Log-Likelihood (lower is better)
    nll = log_loss(y_true, probs_mean, labels=np.arange(NUM_CLASSES))

    # Brier Score (lower is better)
    # One-hot encode true labels
    y_true_onehot = np.eye(NUM_CLASSES)[y_true]
    brier = brier_score_loss(
        y_true_onehot.ravel(),
        probs_mean.ravel(),
        pos_label=1
    )

    # Expected Calibration Error (ECE)
    confidences = np.max(probs_mean, axis=1)
    correct = (y_pred == y_true).astype(int)

    ece = 0.0
    bin_edges = np.linspace(0, 1, num_bins + 1)
    for i in range(num_bins):
        mask = (confidences >= bin_edges[i]) & (confidences < bin_edges[i+1])
        if mask.sum() > 0:
            bin_acc = correct[mask].mean()
            bin_conf = confidences[mask].mean()
            bin_weight = mask.sum() / len(y_true)
            ece += bin_weight * abs(bin_acc - bin_conf)

    # Mean Predictive Entropy (higher = more uncertain)
    entropy = -np.sum(probs_mean * np.log(probs_mean + 1e-10), axis=1)
    mean_entropy = entropy.mean()

    # Uncertainty-Error Correlation
    # Correlation between prediction uncertainty and error
    errors = (y_pred != y_true).astype(int)
    unc_err_corr = np.corrcoef(entropy, errors)[0, 1]

    metrics = {
        'accuracy': acc,
        'nll': nll,
        'brier': brier,
        'ece': ece,
        'mean_entropy': mean_entropy,
        'unc_error_corr': unc_err_corr
    }

    return metrics


def print_metrics(metrics: Dict[str, float], train_time: float, test_time: float):
    """Print evaluation metrics in a formatted table."""
    print(f"\n✓ Accuracy: {metrics['accuracy']:.4f} ({100*metrics['accuracy']:.2f}%)")
    print(f"✓ Negative Log-Likelihood: {metrics['nll']:.4f}")
    print(f"✓ Brier Score: {metrics['brier']:.4f}")
    print(f"✓ Expected Calibration Error: {metrics['ece']:.4f}")
    print(f"✓ Mean Predictive Entropy: {metrics['mean_entropy']:.4f}")
    print(f"✓ Uncertainty-Error Correlation: {metrics['unc_error_corr']:.4f}")
    print(f"\n✓ Training Time: {train_time:.2f} seconds")
    print(f"✓ Test Time: {test_time:.2f} seconds")

    # Sanity checks
    print(f"\n✓ Sanity checks:")
    print(f"  - NLL is finite: {np.isfinite(metrics['nll'])}")
    print(f"  - ECE in [0, 1]: {0 <= metrics['ece'] <= 1}")


## 7. Variational-inference BNN

Each weight has a learnable Gaussian N(μ, σ²) and is sampled with the reparameterisation trick. The loss is cross-entropy plus β·KL, with β = 1/N<sub>train</sub> annealed linearly over the first 10 epochs.

In [ ]:
class BayesianLinear(nn.Module):
    """
    Custom Bayesian Linear layer using reparameterization trick.

    Implements variational inference for neural network weights:
    - Prior: p(w) = N(0, prior_std^2)
    - Posterior: q(w) = N(μ, σ²) (learned)
    - Forward: w = μ + σ * ε, where ε ~ N(0, 1)

    This is the standard approach from Blundell et al. (2015)
    "Weight Uncertainty in Neural Networks".
    """
    def __init__(self, in_features, out_features, prior_std=1.0):
        super(BayesianLinear, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.prior_std = prior_std

        # Weight parameters (mean and log variance)
        self.weight_mu = nn.Parameter(torch.Tensor(out_features, in_features))
        self.weight_log_sigma = nn.Parameter(torch.Tensor(out_features, in_features))

        # Bias parameters (mean and log variance)
        self.bias_mu = nn.Parameter(torch.Tensor(out_features))
        self.bias_log_sigma = nn.Parameter(torch.Tensor(out_features))

        # Initialize parameters
        self.reset_parameters()

    def reset_parameters(self):
        """Initialize variational parameters."""
        # Initialize means with Xavier initialization
        nn.init.xavier_uniform_(self.weight_mu)
        nn.init.zeros_(self.bias_mu)

        # Initialize log sigmas (start with low variance for stability)
        nn.init.constant_(self.weight_log_sigma, -5.0)
        nn.init.constant_(self.bias_log_sigma, -5.0)

    def forward(self, x):
        """
        Forward pass with reparameterization trick.

        Samples weights w = μ + σ * ε and computes linear transformation.
        """
        # Sample weights using reparameterization trick
        weight_sigma = torch.exp(self.weight_log_sigma)
        weight_eps = torch.randn_like(self.weight_mu)
        weight = self.weight_mu + weight_sigma * weight_eps

        # Sample biases
        bias_sigma = torch.exp(self.bias_log_sigma)
        bias_eps = torch.randn_like(self.bias_mu)
        bias = self.bias_mu + bias_sigma * bias_eps

        # Linear transformation
        output = F.linear(x, weight, bias)
        return output

    def kl_loss(self):
        """
        Compute KL divergence between posterior q(w) and prior p(w).

        For Gaussian distributions, this has a closed form:
        KL(q||p) = log(σ_prior/σ_post) + (σ_post² + μ_post²)/(2σ_prior²) - 0.5

        Returns:
        --------
        kl : torch.Tensor
            Scalar KL divergence
        """
        # KL for weights
        weight_sigma = torch.exp(self.weight_log_sigma)
        weight_kl = torch.log(self.prior_std / weight_sigma) + \
                    (weight_sigma**2 + self.weight_mu**2) / (2 * self.prior_std**2) - 0.5

        # KL for biases
        bias_sigma = torch.exp(self.bias_log_sigma)
        bias_kl = torch.log(self.prior_std / bias_sigma) + \
                  (bias_sigma**2 + self.bias_mu**2) / (2 * self.prior_std**2) - 0.5

        return weight_kl.sum() + bias_kl.sum()


class VIBNN(nn.Module):
    """
    Variational Inference Bayesian Neural Network.

    Architecture: Input → 64 → 32 → 4 (logits)

    Uses custom Bayesian linear layers with reparameterization trick.
    Training optimizes ELBO: minimize NLL + β * KL divergence.
    """

    def __init__(self, input_dim, hidden_dim_1=64, hidden_dim_2=32, output_dim=4):
        super(VIBNN, self).__init__()

        # Use custom Bayesian layers
        self.fc1 = BayesianLinear(input_dim, hidden_dim_1)
        self.fc2 = BayesianLinear(hidden_dim_1, hidden_dim_2)
        self.fc3 = BayesianLinear(hidden_dim_2, output_dim)
        self.activation = nn.ReLU()

    def forward(self, x):
        """Forward pass through Bayesian network."""
        x = self.activation(self.fc1(x))
        x = self.activation(self.fc2(x))
        logits = self.fc3(x)
        return logits

    def kl_divergence(self):
        """Compute total KL divergence for all Bayesian layers."""
        kl = self.fc1.kl_loss() + self.fc2.kl_loss() + self.fc3.kl_loss()
        return kl


Training uses Adam (learning rate 1e-3) for up to 50 epochs, with early stopping on validation loss (patience 10). Prediction keeps the layers stochastic and averages 50 forward passes.

In [ ]:
def train_vi_bnn(model, train_loader, val_loader, optimizer, epochs=50,
                 device='cpu', kl_weight=1.0, patience=10, kl_anneal_epochs=10):
    """
    Train Variational Inference BNN with KL annealing.

    Loss: ELBO = NLL + annealed_kl_weight * KL

    Parameters:
    -----------
    model : VIBNN
        Variational BNN model
    train_loader : DataLoader
        Training data loader
    val_loader : DataLoader
        Validation data loader
    optimizer : torch.optim.Optimizer
        Optimizer (e.g., Adam)
    epochs : int
        Number of training epochs
    device : str or torch.device
        Device for training
    kl_weight : float
        FINAL weight for KL divergence term (1/N_train recommended)
    patience : int
        Early stopping patience
    kl_anneal_epochs : int
        Number of epochs to linearly anneal KL weight from 0 to kl_weight

    Returns:
    --------
    model : VIBNN
        Trained model (best validation checkpoint)
    train_time : float
        Total training time in seconds
    train_history : dict
        Training history (losses and accuracies)
    """
    print(f"\n" + "="*80)
    print("TRAINING VARIATIONAL BNN")
    print("="*80)
    print(f"✓ KL weight (final): {kl_weight:.6e}")
    print(f"✓ KL annealing epochs: {kl_anneal_epochs}")

    model = model.to(device)
    criterion = nn.CrossEntropyLoss()

    best_val_loss = float('inf')
    best_model_state = None
    epochs_without_improvement = 0

    train_history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': []
    }

    start_time = time.perf_counter()

    for epoch in range(epochs):
        # KL annealing: linearly warm up from 0 to kl_weight
        annealed_kl_weight = kl_weight * min(1.0, (epoch + 1) / kl_anneal_epochs)

        # Training phase
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for batch_idx, (X_batch, y_batch) in enumerate(train_loader):
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            # Forward pass
            logits = model(X_batch)

            # Compute NLL
            nll = criterion(logits, y_batch)

            # Compute KL divergence
            kl = model.kl_divergence()

            # ELBO loss with annealed KL weight
            loss = nll + annealed_kl_weight * kl

            # Backward pass
            loss.backward()
            optimizer.step()

            # Track metrics
            train_loss += loss.item()
            _, predicted = torch.max(logits, 1)
            train_correct += (predicted == y_batch).sum().item()
            train_total += y_batch.size(0)

        # Average training metrics
        avg_train_loss = train_loss / len(train_loader)
        train_acc = 100.0 * train_correct / train_total

        # Validation phase (use full KL weight)
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)

                logits = model(X_batch)
                nll = criterion(logits, y_batch)
                kl = model.kl_divergence()
                loss = nll + kl_weight * kl  # Full weight for validation

                val_loss += loss.item()
                _, predicted = torch.max(logits, 1)
                val_correct += (predicted == y_batch).sum().item()
                val_total += y_batch.size(0)

        # Average validation metrics
        avg_val_loss = val_loss / len(val_loader)
        val_acc = 100.0 * val_correct / val_total

        # Store history
        train_history['train_loss'].append(avg_train_loss)
        train_history['train_acc'].append(train_acc)
        train_history['val_loss'].append(avg_val_loss)
        train_history['val_acc'].append(val_acc)

        # Print progress
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch [{epoch+1:3d}/{epochs}] | "
                  f"KL wt: {annealed_kl_weight:.2e} | "
                  f"Train Loss: {avg_train_loss:.4f}, Acc: {train_acc:.2f}% | "
                  f"Val Loss: {avg_val_loss:.4f}, Acc: {val_acc:.2f}%")

        # Early stopping
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_state = model.state_dict().copy()
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            print(f"\n✓ Early stopping at epoch {epoch+1}")
            break

    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    train_time = time.perf_counter() - start_time

    print(f"\n✓ Training completed in {train_time:.2f} seconds")
    print(f"✓ Best validation loss: {best_val_loss:.4f}")

    return model, train_time, train_history


def predict_vi_bnn(model, data_loader, device='cpu', num_mc_samples=50):
    """
    Generate predictions using Variational BNN with Monte Carlo sampling.

    Parameters:
    -----------
    model : VIBNN
        Trained VI-BNN model
    data_loader : DataLoader
        Data to predict on
    device : str or torch.device
        Device for inference
    num_mc_samples : int
        Number of Monte Carlo forward passes

    Returns:
    --------
    probs_samples : np.ndarray
        Probability samples (num_mc_samples, N, num_classes)
    probs_mean : np.ndarray
        Mean probabilities (N, num_classes)
    inference_time : float
        Time taken for inference
    """
    model = model.to(device)
    model.train()  # Keep in training mode for stochastic forward passes

    all_probs_samples = []

    start_time = time.perf_counter()

    # Collect predictions from multiple MC samples
    for mc_idx in range(num_mc_samples):
        batch_probs = []

        with torch.no_grad():
            for X_batch, _ in data_loader:
                X_batch = X_batch.to(device)
                logits = model(X_batch)
                probs = F.softmax(logits, dim=1)
                batch_probs.append(probs.cpu().numpy())

        # Concatenate batches
        sample_probs = np.concatenate(batch_probs, axis=0)
        all_probs_samples.append(sample_probs)

    inference_time = time.perf_counter() - start_time

    # Stack samples: (num_mc_samples, N, num_classes)
    probs_samples = np.stack(all_probs_samples, axis=0)

    # Compute mean: (N, num_classes)
    probs_mean = np.mean(probs_samples, axis=0)

    # Verify probability constraints
    assert np.allclose(probs_mean.sum(axis=1), 1.0, atol=1e-3), \
        "Probabilities do not sum to 1"

    return probs_samples, probs_mean, inference_time


## 8. MCMC BNN (Pyro NUTS)

This uses the same architecture with N(0, 1) priors on every weight and bias. NUTS runs one chain with target acceptance 0.8, 200 warm-up steps and 150 posterior samples. It uses a random 5,000-cell subsample of the original (pre-SMOTE) training data. SMOTE points are synthetic and would distort the likelihood, and full-data NUTS would take too long. Predictions are the Bayesian model average over the posterior samples.

In [ ]:
class MCMCMLPModel(nn.Module):
    """
    Standard MLP for use with Pyro MCMC.

    Architecture: Input → 64 → 32 → 4 (logits)
    """
    def __init__(self, input_dim, hidden_dim_1=64, hidden_dim_2=32, output_dim=4):
        super(MCMCMLPModel, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim_1)
        self.fc2 = nn.Linear(hidden_dim_1, hidden_dim_2)
        self.fc3 = nn.Linear(hidden_dim_2, output_dim)
        self.activation = nn.ReLU()

    def forward(self, x):
        """Forward pass."""
        x = self.activation(self.fc1(x))
        x = self.activation(self.fc2(x))
        logits = self.fc3(x)
        return logits


def pyro_model(X, y, hidden_dim_1=64, hidden_dim_2=32):
    """
    Pyro probabilistic model for MCMC inference.

    Places Gaussian priors on all weights and biases.

    Parameters:
    -----------
    X : torch.Tensor
        Input features (N, input_dim)
    y : torch.Tensor or None
        Target labels (N,). If None, generates samples from prior.
    hidden_dim_1, hidden_dim_2 : int
        Hidden layer dimensions
    """
    input_dim = X.shape[1]
    output_dim = NUM_CLASSES

    # Priors on layer 1
    fc1_w_prior = dist.Normal(0., 1.).expand([hidden_dim_1, input_dim]).to_event(2)
    fc1_b_prior = dist.Normal(0., 1.).expand([hidden_dim_1]).to_event(1)
    fc1_w = pyro.sample("fc1_w", fc1_w_prior)
    fc1_b = pyro.sample("fc1_b", fc1_b_prior)

    # Priors on layer 2
    fc2_w_prior = dist.Normal(0., 1.).expand([hidden_dim_2, hidden_dim_1]).to_event(2)
    fc2_b_prior = dist.Normal(0., 1.).expand([hidden_dim_2]).to_event(1)
    fc2_w = pyro.sample("fc2_w", fc2_w_prior)
    fc2_b = pyro.sample("fc2_b", fc2_b_prior)

    # Priors on layer 3
    fc3_w_prior = dist.Normal(0., 1.).expand([output_dim, hidden_dim_2]).to_event(2)
    fc3_b_prior = dist.Normal(0., 1.).expand([output_dim]).to_event(1)
    fc3_w = pyro.sample("fc3_w", fc3_w_prior)
    fc3_b = pyro.sample("fc3_b", fc3_b_prior)

    # Forward pass
    h1 = F.relu(F.linear(X, fc1_w, fc1_b))
    h2 = F.relu(F.linear(h1, fc2_w, fc2_b))
    logits = F.linear(h2, fc3_w, fc3_b)

    # Likelihood
    with pyro.plate("data", X.shape[0]):
        obs = pyro.sample("obs", dist.Categorical(logits=logits), obs=y)

    return logits


def run_mcmc(X_train, y_train, input_dim,
             warmup_steps=200, num_samples=300, device='cpu'):
    """
    Run MCMC sampling using Pyro's NUTS sampler.

    Expects a subsample of the original (pre-SMOTE) training data.

    Parameters:
    -----------
    X_train : np.ndarray
        Training features (subsampled, pre-SMOTE)
    y_train : np.ndarray
        Training labels
    input_dim : int
        Number of input features
    warmup_steps : int
        Number of MCMC warmup/burn-in steps
    num_samples : int
        Number of posterior samples to draw
    device : str or torch.device
        Device for computation

    Returns:
    --------
    posterior_samples : dict
        Dictionary of parameter samples from posterior
    mcmc_time : float
        Total MCMC runtime
    mlp_model : MCMCMLPModel
        MLP model for prediction
    """
    print(f"\n{'='*80}")
    print("RUNNING MCMC WITH PYRO (NUTS SAMPLER)")
    print(f"{'='*80}")

    print(f"\n✓ MCMC configuration:")
    print(f"  Training samples: {len(X_train):,}")
    print(f"  Warmup steps: {warmup_steps}")
    print(f"  Posterior samples: {num_samples}")
    print(f"  Architecture: {input_dim} → {HIDDEN_DIM_1} → {HIDDEN_DIM_2} → {NUM_CLASSES}")

    # Convert to tensors
    X_train_t = torch.FloatTensor(X_train).to(device)
    y_train_t = torch.LongTensor(y_train).to(device)

    # Setup NUTS sampler
    nuts_kernel = NUTS(pyro_model, adapt_step_size=True, target_accept_prob=0.8)
    mcmc = MCMC(
        nuts_kernel,
        num_samples=num_samples,
        warmup_steps=warmup_steps,
        num_chains=1
    )

    # Run MCMC
    print(f"\n✓ Starting MCMC sampling...")
    start_time = time.perf_counter()

    mcmc.run(X_train_t, y_train_t, HIDDEN_DIM_1, HIDDEN_DIM_2)

    mcmc_time = time.perf_counter() - start_time

    print(f"\n✓ MCMC completed in {mcmc_time:.2f} seconds")

    # Extract posterior samples
    posterior_samples = mcmc.get_samples()

    print(f"\n✓ Posterior samples extracted:")
    for param_name, param_samples in posterior_samples.items():
        print(f"  {param_name}: {param_samples.shape}")

    # Create MLP model for prediction
    mlp_model = MCMCMLPModel(input_dim, HIDDEN_DIM_1, HIDDEN_DIM_2, NUM_CLASSES)

    return posterior_samples, mcmc_time, mlp_model


def predict_mcmc_bnn(posterior_samples, mlp_model, data_loader, device='cpu'):
    """
    Generate predictions using MCMC posterior samples.

    Parameters:
    -----------
    posterior_samples : dict
        Posterior parameter samples from MCMC
    mlp_model : MCMCMLPModel
        MLP model for forward passes
    data_loader : DataLoader
        Data to predict on
    device : str or torch.device
        Device for inference

    Returns:
    --------
    probs_samples : np.ndarray
        Probability samples (num_samples, N, num_classes)
    probs_mean : np.ndarray
        Mean probabilities (N, num_classes)
    inference_time : float
        Time taken for inference
    """
    mlp_model = mlp_model.to(device)
    mlp_model.eval()

    num_samples = posterior_samples['fc1_w'].shape[0]

    # Collect all test data
    X_test_list = []
    for X_batch, _ in data_loader:
        X_test_list.append(X_batch)
    X_test_full = torch.cat(X_test_list, dim=0).to(device)

    all_probs_samples = []

    start_time = time.perf_counter()

    # Sample from posterior
    for sample_idx in range(num_samples):
        # Load parameters from this posterior sample
        state_dict = {
            'fc1.weight': posterior_samples['fc1_w'][sample_idx],
            'fc1.bias': posterior_samples['fc1_b'][sample_idx],
            'fc2.weight': posterior_samples['fc2_w'][sample_idx],
            'fc2.bias': posterior_samples['fc2_b'][sample_idx],
            'fc3.weight': posterior_samples['fc3_w'][sample_idx],
            'fc3.bias': posterior_samples['fc3_b'][sample_idx]
        }
        mlp_model.load_state_dict(state_dict)

        # Forward pass
        with torch.no_grad():
            logits = mlp_model(X_test_full)
            probs = F.softmax(logits, dim=1)
            all_probs_samples.append(probs.cpu().numpy())

    inference_time = time.perf_counter() - start_time

    # Stack samples: (num_samples, N, num_classes)
    probs_samples = np.stack(all_probs_samples, axis=0)

    # Compute mean: (N, num_classes)
    probs_mean = np.mean(probs_samples, axis=0)

    # Verify probability constraints
    assert np.allclose(probs_mean.sum(axis=1), 1.0, atol=1e-3), \
        "Probabilities do not sum to 1"

    return probs_samples, probs_mean, inference_time


## 9. Experiment runners

For one hazard, the runner splits and resamples the data, trains and evaluates the VI-BNN, then runs NUTS and evaluates the MCMC-BNN. The full runner loops over the three hazards and writes `outputs/bnn_results.csv`.

In [ ]:
def run_single_hazard_experiment(
    df: pd.DataFrame,
    hazard_col: str,
    device: torch.device = DEVICE
) -> Dict[str, Any]:
    """
    Run complete VI-BNN and MCMC-BNN experiment for a single hazard.

    Design choices:
    1. VI KL weight = 1/N_train, linearly annealed over the first epochs
    2. MCMC runs on a 5k subsample of the original (pre-SMOTE) training data

    Parameters:
    -----------
    df : pd.DataFrame
        Full dataset with features and hazard column
    hazard_col : str
        Name of hazard column to use as target
    device : torch.device
        Device for training

    Returns:
    --------
    results : dict
        Complete results for this hazard
    """
    print(f"\n\n" + "="*80)
    print(f"EXPERIMENT: {hazard_col}")
    print("="*80)

    # Extract features and target
    feature_cols = [col for col in df.columns if col not in HAZARDS]
    X = df[feature_cols].values
    y = df[hazard_col].values

    input_dim = X.shape[1]

    print(f"\n✓ Dataset: {len(X):,} samples, {input_dim} features")
    print(f"✓ Target: {hazard_col}")

    # Split and resample (returns the resampled and the original training data)
    X_train_res, y_train_res, X_val, y_val, X_test, y_test, X_train_orig, y_train_orig = \
        split_and_resample(X, y)

    # Create DataLoaders (use RESAMPLED data for VI-BNN)
    train_loader, val_loader, test_loader = create_dataloaders(
        X_train_res, y_train_res, X_val, y_val, X_test, y_test,
        batch_size=VI_BATCH_SIZE
    )

    # =========================================================================
    # TRAIN VI-BNN
    # =========================================================================
    print(f"\n{'='*80}")
    print("TRAINING VI-BNN")
    print(f"{'='*80}")

    vi_model = VIBNN(input_dim, HIDDEN_DIM_1, HIDDEN_DIM_2, NUM_CLASSES)
    vi_optimizer = torch.optim.Adam(vi_model.parameters(), lr=VI_LEARNING_RATE)

    # KL weight = 1/N_train
    kl_weight = 1.0 / len(X_train_res)
    print(f"\n✓ KL weight: 1/{len(X_train_res):,} = {kl_weight:.6e}")

    vi_model, vi_train_time, vi_history = train_vi_bnn(
        vi_model, train_loader, val_loader, vi_optimizer,
        epochs=VI_EPOCHS, device=device, kl_weight=kl_weight,
        patience=VI_PATIENCE, kl_anneal_epochs=VI_KL_ANNEAL_EPOCHS
    )

    print(f"\n✓ Generating VI-BNN predictions on test set...")
    vi_probs_samples, vi_probs_mean, vi_test_time = predict_vi_bnn(
        vi_model, test_loader, device=device, num_mc_samples=VI_MC_SAMPLES
    )

    print(f"\n{'='*80}")
    print(f"EVALUATING VI-BNN ({hazard_col})")
    print(f"{'='*80}")

    vi_metrics = evaluate_predictions(y_test, vi_probs_samples, vi_probs_mean)
    print_metrics(vi_metrics, vi_train_time, vi_test_time)

    # Sanity check probabilities
    probs_sum_check = np.allclose(vi_probs_mean.sum(axis=1), 1.0, atol=1e-3)
    print(f"  - Probs sum to ~1: {probs_sum_check}")

    # =========================================================================
    # TRAIN MCMC-BNN (subsample of the original pre-SMOTE training data)
    # =========================================================================
    print(f"\n{'='*80}")
    print("TRAINING MCMC-BNN")
    print(f"{'='*80}")

    # Subsample from the original pre-SMOTE training data
    print(f"\n✓ Subsampling {MCMC_SUBSET_SIZE:,} samples from ORIGINAL training set")
    print(f"  (Original training set: {len(X_train_orig):,} samples)")

    rng = np.random.default_rng(SEED)
    mcmc_idx = rng.choice(len(X_train_orig), size=MCMC_SUBSET_SIZE, replace=False)
    X_mcmc = X_train_orig[mcmc_idx]
    y_mcmc = y_train_orig[mcmc_idx]

    print(f"\n✓ MCMC subset:")
    print(f"  Size: {len(X_mcmc):,} samples")
    unique, counts = np.unique(y_mcmc, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:6d} ({100*count/len(y_mcmc):5.2f}%)")

    posterior_samples, mcmc_train_time, mcmc_mlp = run_mcmc(
        X_mcmc, y_mcmc, input_dim,
        warmup_steps=MCMC_WARMUP, num_samples=MCMC_SAMPLES, device=device
    )

    print(f"\n✓ Generating MCMC-BNN predictions on test set...")
    mcmc_probs_samples, mcmc_probs_mean, mcmc_test_time = predict_mcmc_bnn(
        posterior_samples, mcmc_mlp, test_loader, device=device
    )

    print(f"\n{'='*80}")
    print(f"EVALUATING MCMC-BNN ({hazard_col})")
    print(f"{'='*80}")

    mcmc_metrics = evaluate_predictions(y_test, mcmc_probs_samples, mcmc_probs_mean)
    print_metrics(mcmc_metrics, mcmc_train_time, mcmc_test_time)

    # Sanity check probabilities
    probs_sum_check = np.allclose(mcmc_probs_mean.sum(axis=1), 1.0, atol=1e-3)
    print(f"  - Probs sum to ~1: {probs_sum_check}")

    # =========================================================================
    # Compile results
    # =========================================================================
    results = {
        'hazard': hazard_col,
        'vi_metrics': vi_metrics,
        'vi_train_time': vi_train_time,
        'vi_test_time': vi_test_time,
        'mcmc_metrics': mcmc_metrics,
        'mcmc_train_time': mcmc_train_time,
        'mcmc_test_time': mcmc_test_time,
        'test_size': len(y_test)
    }

    return results


In [ ]:
def run_full_experiment(csv_path: str = CSV_PATH) -> pd.DataFrame:
    """
    Run complete experiment across all 3 hazards.

    Returns:
    --------
    results_df : pd.DataFrame
        Combined results in tabular format
    """
    print(f"\n\n" + "="*80)
    print("STARTING FULL EXPERIMENT: 3 HAZARDS × 2 MODELS = 6 TOTAL MODELS")
    print(f"{'='*80}")

    # Load and preprocess data
    df = load_and_preprocess_data(csv_path)

    all_results = []

    # Run experiment for each hazard
    for hazard_col in HAZARDS:
        results = run_single_hazard_experiment(df, hazard_col, device=DEVICE)
        all_results.append(results)

    # Convert to DataFrame
    rows = []
    for res in all_results:
        # VI-BNN row
        rows.append({
            'hazard': res['hazard'],
            'model': 'VI-BNN',
            'accuracy': res['vi_metrics']['accuracy'],
            'nll': res['vi_metrics']['nll'],
            'brier': res['vi_metrics']['brier'],
            'ece': res['vi_metrics']['ece'],
            'mean_entropy': res['vi_metrics']['mean_entropy'],
            'unc_error_corr': res['vi_metrics']['unc_error_corr'],
            'train_time': res['vi_train_time'],
            'test_time': res['vi_test_time'],
            'test_size': res['test_size']
        })

        # MCMC-BNN row
        rows.append({
            'hazard': res['hazard'],
            'model': 'MCMC-BNN',
            'accuracy': res['mcmc_metrics']['accuracy'],
            'nll': res['mcmc_metrics']['nll'],
            'brier': res['mcmc_metrics']['brier'],
            'ece': res['mcmc_metrics']['ece'],
            'mean_entropy': res['mcmc_metrics']['mean_entropy'],
            'unc_error_corr': res['mcmc_metrics']['unc_error_corr'],
            'train_time': res['mcmc_train_time'],
            'test_time': res['mcmc_test_time'],
            'test_size': res['test_size']
        })

    results_df = pd.DataFrame(rows)

    # Save to CSV
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    output_path = os.path.join(OUTPUT_DIR, "bnn_results.csv")
    results_df.to_csv(output_path, index=False)

    print(f"\n\n" + "="*80)
    print("EXPERIMENT COMPLETE")
    print(f"{'='*80}")
    print(f"\n✓ Results saved to: {output_path}")
    print(f"\n{results_df.to_string(index=False)}")

    return results_df


## 10. Run all three hazards

**Long-running cell:** expect several hours on CPU, mostly NUTS. Use `SMOKE_TEST = True` for a quick check.

In [ ]:
results_df = run_full_experiment(CSV_PATH)

print(f"\n\n{'='*80}")
print("FINAL SUMMARY")
print(f"{'='*80}")

# Summary statistics
print(f"\n✓ Total models trained: {len(results_df)}")
print(f"\n✓ Average metrics by model type:")
summary = results_df.groupby('model')[['accuracy', 'nll', 'ece', 'train_time']].mean()
print(f"\n{summary}")

print(f"\n{'='*80}")
print("ALL EXPERIMENTS COMPLETED SUCCESSFULLY")
print(f"{'='*80}\n")


## 11. Uncertainty analysis (VI-BNN)

For each hazard, this section retrains the VI-BNN on the same split. It splits predictive entropy into an aleatoric part (expected entropy) and an epistemic part (mutual information), and saves the test-set probabilities to `outputs/analysis_outputs/` as `.npy` files.

In [ ]:
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

SAVE_DIR = f"{OUTPUT_DIR}/analysis_outputs/"
os.makedirs(SAVE_DIR, exist_ok=True)

df_analysis = load_and_preprocess_data(CSV_PATH)
FEATURE_COLS = [col for col in df_analysis.columns if col not in HAZARDS]

analysis_data = {}

for hazard_col in HAZARDS:
    hkey = hazard_col.replace(" ", "_").replace("/", "_").lower()
    print(f"\n{'='*60}\nANALYSIS PREP: {hazard_col}\n{'='*60}")

    X_raw = df_analysis[FEATURE_COLS].values
    y_raw = df_analysis[hazard_col].values

    # Same stratified 80/20 split as split_and_resample, so X_test_orig_df lines up
    # row-for-row with X_test (unscaled features are kept for plotting)
    _, test_local_idx = train_test_split(
        np.arange(len(X_raw)),
        test_size=0.2,
        random_state=SEED,
        stratify=y_raw
    )
    X_test_orig_df = df_analysis.iloc[test_local_idx][FEATURE_COLS].reset_index(drop=True)

    # Rebuild identical splits for VI retraining
    (X_train_res, y_train_res,
     X_val, y_val,
     X_test, y_test,
     X_train_orig, y_train_orig) = split_and_resample(X_raw, y_raw)

    print(f"✓ X_test size:        {len(X_test)}")
    print(f"✓ X_test_orig_df size:{len(X_test_orig_df)}  ← must match above")
    assert len(X_test) == len(X_test_orig_df), \
        f"Size mismatch: {len(X_test)} vs {len(X_test_orig_df)}"

    input_dim = X_test.shape[1]

    train_loader, val_loader, test_loader = create_dataloaders(
        X_train_res, y_train_res, X_val, y_val, X_test, y_test,
        batch_size=VI_BATCH_SIZE
    )

    # Retrain the VI-BNN on the same split (about 1-2 min per hazard on CPU)
    kl_weight    = 1.0 / len(X_train_res)
    vi_model_a   = VIBNN(input_dim, HIDDEN_DIM_1, HIDDEN_DIM_2, NUM_CLASSES)
    vi_optimizer_a = torch.optim.Adam(vi_model_a.parameters(), lr=VI_LEARNING_RATE)
    vi_model_a, _, _ = train_vi_bnn(
        vi_model_a, train_loader, val_loader, vi_optimizer_a,
        epochs=VI_EPOCHS, device=DEVICE,
        kl_weight=kl_weight, patience=VI_PATIENCE,
        kl_anneal_epochs=VI_KL_ANNEAL_EPOCHS
    )

    vi_probs_samples, vi_probs_mean, _ = predict_vi_bnn(
        vi_model_a, test_loader, device=DEVICE, num_mc_samples=VI_MC_SAMPLES
    )

    np.save(f"{SAVE_DIR}vi_probs_mean_{hkey}.npy",    vi_probs_mean)
    np.save(f"{SAVE_DIR}vi_probs_samples_{hkey}.npy", vi_probs_samples)
    np.save(f"{SAVE_DIR}y_test_{hkey}.npy",           y_test)

    vi_entropy = -np.sum(vi_probs_mean * np.log(vi_probs_mean + 1e-10), axis=1)

    mean_of_entropies = np.mean(
        [-np.sum(s * np.log(s + 1e-10), axis=1) for s in vi_probs_samples], axis=0
    )
    epistemic_ent = vi_entropy - mean_of_entropies
    aleatoric_ent = mean_of_entropies

    analysis_data[hazard_col] = {
        'hkey':           hkey,
        'y_test':         y_test,
        'X_test_orig_df': X_test_orig_df,
        'vi_entropy':     vi_entropy,
        'epistemic_ent':  epistemic_ent,
        'aleatoric_ent':  aleatoric_ent,
        'vi_probs_mean':  vi_probs_mean,
    }

## 12. Figures

Three figures are produced for each hazard:

- VI predictive entropy against elevation and distance to river, with Spearman ρ;
- the epistemic/aleatoric decomposition;
- a VI reliability diagram over all class probabilities.

Figures and `entropy_feature_correlations.csv` are saved to `outputs/analysis_outputs/`.

In [ ]:
def make_analysis_plots(hazard_col, data, save_dir):
    hkey          = data['hkey']
    y_test        = data['y_test']
    X_df          = data['X_test_orig_df']
    vi_entropy    = data['vi_entropy']
    epistemic_ent = data['epistemic_ent']
    aleatoric_ent = data['aleatoric_ent']
    vi_probs_mean = data['vi_probs_mean']

    elevation     = X_df['ELEVATION'].values
    dist_river    = X_df['DISTANCE TO RIVER'].values

    # ── PLOT 1: Predictive entropy vs flood-prone features ──
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(f"VI-BNN Predictive Entropy vs Flood Features\n({hazard_col})",
                 fontweight='bold')

    for ax, (feat, fname) in zip(
        axes,
        [(elevation, "Elevation (m)"), (dist_river, "Distance to River (m)")]
    ):
        rho, pval = spearmanr(feat, vi_entropy)
        sc = ax.scatter(feat, vi_entropy, c=y_test, cmap='RdYlGn_r',
                        alpha=0.3, s=6, rasterized=True)
        ax.set_xlabel(fname)
        ax.set_ylabel("Predictive Entropy H[y|x]")
        ax.set_title(f"Spearman ρ = {rho:.3f}  (p = {pval:.2e})")
        plt.colorbar(sc, ax=ax, label="True Hazard Class")

    plt.tight_layout()
    path = f"{save_dir}entropy_vs_features_{hkey}.png"
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved: {path}")

    # Spearman correlations for report table
    rho_elev,  _ = spearmanr(elevation,  vi_entropy)
    rho_river, _ = spearmanr(dist_river, vi_entropy)
    print(f"  → VI entropy ~ Elevation:        ρ = {rho_elev:.3f}")
    print(f"  → VI entropy ~ Dist-to-River:    ρ = {rho_river:.3f}")

    # ── PLOT 2: Epistemic vs Aleatoric decomposition ────────
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    fig.suptitle(
        f"VI-BNN Uncertainty Decomposition: Epistemic vs Aleatoric\n({hazard_col})",
        fontweight='bold'
    )
    scatter_kw = dict(alpha=0.3, s=6, rasterized=True, cmap='RdYlGn_r')
    labels_unc = [("Epistemic (MI)", epistemic_ent), ("Aleatoric", aleatoric_ent)]

    for row, (unc_label, unc_arr) in enumerate(labels_unc):
        for col, (feat, fname) in enumerate(
            [(elevation, "Elevation (m)"), (dist_river, "Distance to River (m)")]
        ):
            ax = axes[row, col]
            sc = ax.scatter(feat, unc_arr, c=y_test, **scatter_kw)
            ax.set_xlabel(fname)
            ax.set_ylabel(unc_label)
            ax.set_title(f"{unc_label} vs {fname.split('(')[0].strip()}")
            plt.colorbar(sc, ax=ax, label="True Class")

    plt.tight_layout()
    path = f"{save_dir}uncertainty_decomp_{hkey}.png"
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved: {path}")

    # ── PLOT 3: Calibration reliability diagram (VI only) ───
    n_bins = 10
    bin_edges = np.linspace(0, 1, n_bins + 1)
    y_onehot  = np.eye(NUM_CLASSES)[y_test]
    probs_flat  = vi_probs_mean.ravel()
    labels_flat = y_onehot.ravel()

    bin_accs, bin_confs = [], []
    for i in range(n_bins):
        mask = (probs_flat >= bin_edges[i]) & (probs_flat < bin_edges[i+1])
        if mask.sum() > 0:
            bin_accs.append(labels_flat[mask].mean())
            bin_confs.append(probs_flat[mask].mean())

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.bar(bin_confs, bin_accs, width=1/n_bins, alpha=0.6,
           color='steelblue', edgecolor='white', label='VI-BNN')
    ax.plot([0, 1], [0, 1], 'r--', lw=1.5, label='Perfect calibration')
    ax.set_xlabel("Confidence")
    ax.set_ylabel("Accuracy")
    ax.set_title(f"Reliability Diagram — VI-BNN\n({hazard_col})")
    ax.legend()
    plt.tight_layout()
    path = f"{save_dir}calibration_vi_{hkey}.png"
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved: {path}")

    return {'rho_elevation': rho_elev, 'rho_dist_river': rho_river}


# ── Run plots for all 3 hazards ──────────────────────────────
corr_rows = []
for hazard_col, data in analysis_data.items():
    print(f"\n{'─'*50}\nGenerating plots: {hazard_col}\n{'─'*50}")
    row = make_analysis_plots(hazard_col, data, SAVE_DIR)
    row['hazard'] = hazard_col
    corr_rows.append(row)

# ── Summary correlation table ──────
corr_df = pd.DataFrame(corr_rows)[['hazard', 'rho_elevation', 'rho_dist_river']]
corr_df.to_csv(f"{SAVE_DIR}entropy_feature_correlations.csv", index=False)
print("\n✓ entropy_feature_correlations.csv saved")
print(corr_df.to_string(index=False))